<a href="https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growing content and content age

The paper reports that growing content was younger and longer on average than declining content. Growing pages averaged about 184 days old and 3,180 words, while declining pages averaged about 230 days old and 2,311 words.

**Methodology question:** Where does the declining/growing outcome come from, and how are the performance windows defined relative to content age? I would check that the comparison is being used as observational evidence and that the methodology does not imply that content age itself causes growth or decline. I would also check whether the validation design supports the strength of the claim.

### Finding 2 — AI traffic behaves differently

The paper reports that pages with high AI referral traffic had substantially more impressions but a weaker average Google position than pages with no AI referral traffic. The paper presents this as evidence that AI-referral visibility behaves differently from traditional organic search visibility.

**Methodology question:** How was the high-AI group defined, and does the validation/design account for other differences between the groups, such as content age, visibility, or the active-content population? I would check these factors before interpreting the observed difference as a broader behavioral effect.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/FlyRank-Internship/week1-flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
print("Columns in df:")
print(df.columns.tolist())

print("\nRows:", len(df))

Columns in df:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Rows: 30000


## 2. My model under an honest split (before/after)

### Before vs. after honest validation

The Week-5 Decision Tree was evaluated again using a grouped-by-client split, with no client overlap between training and test data. The original ML-08 test results were compared with the ML-09 grouped-split results.

| Metric | ML-08 | ML-09 grouped split |
|---|---:|---:|
| Accuracy | 0.5776 | 0.5671 |
| Precision | 0.5855 | 0.5703 |
| Recall | 0.5938 | 0.6199 |
| F1 | 0.5896 | 0.5940 |

The grouped re-run showed slightly lower measured accuracy and precision, while recall and F1 were slightly higher. This indicates a change in the precision-recall trade-off rather than a clear overall improvement or deterioration. The test declining rate was 0.511 in the grouped evaluation.*

In [ ]:
# Section 2 — Recreate the Week-5 target

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Label distribution:")
print(df["is_declining_label"].value_counts())

print("\nDeclining rate:",
      round(df["is_declining_label"].mean(), 4))

Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.5421


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=groups
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("\nTotal rows:", len(df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("Client overlap:", len(train_clients & test_clients))

print("\nTrain declining rate:",
      round(train_df["is_declining_label"].mean(), 4))

print("Test declining rate:",
      round(test_df["is_declining_label"].mean(), 4))


Total rows: 30000
Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7
Client overlap: 0

Train declining rate: 0.5501
Test declining rate: 0.511


In [ ]:
# Section 2 — Re-run the Week-5 Decision Tree on the honest grouped split

from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

X_train = train_df[model_features].copy()
X_test = test_df[model_features].copy()

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

# Fit imputation ONLY on training data
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Same model type as ML-08
tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

tree.fit(X_train_imp, y_train)

y_pred = tree.predict(X_test_imp)

# Metrics
honest_accuracy = accuracy_score(y_test, y_pred)
honest_precision = precision_score(y_test, y_pred, zero_division=0)
honest_recall = recall_score(y_test, y_pred, zero_division=0)
honest_f1 = f1_score(y_test, y_pred, zero_division=0)

print("Honest grouped-split Decision Tree results")
print("--------------------------------------------")
print("Accuracy :", round(honest_accuracy, 4))
print("Precision:", round(honest_precision, 4))
print("Recall   :", round(honest_recall, 4))
print("F1       :", round(honest_f1, 4))
print("Test base rate:", round(y_test.mean(), 4))

Honest grouped-split Decision Tree results
--------------------------------------------
Accuracy : 0.5671
Precision: 0.5703
Recall   : 0.6199
F1       : 0.594
Test base rate: 0.511


### 3. Leakage audit

I audited the final ten Decision Tree features against the target and the baseline/product-flag fields.

The target `is_declining_label` is derived from `trend_direction`, so `trend_direction` and `trend_pct` were excluded from the model features. No baseline-related or product-flag columns were present in the final feature set.

I also checked the feature names for obvious future or target-derived fields. The remaining concern is temporal window alignment: aggregate features such as `impressions_90d`, `clicks_90d`, and `sessions_90d` must represent information available before the prediction outcome window. The starter dataset does not expose a separate prediction-date column in the 30,000-row frame, so this notebook cannot independently prove the exact temporal boundary of every aggregate. This is therefore treated as a limitation rather than claimed as fully verified.

The audit found no label-derived or baseline/product-flag features in the final feature set. Temporal leakage could not be completely verified from the available starter-frame columns.*

In [ ]:
# Section 3 — Leakage audit

final_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

target_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

baseline_keywords = [
    "baseline",
    "score",
    "flag",
    "action",
    "reason"
]

print("Final model features:")
print(final_features)

print("\n1. Label-derived feature check")
label_leaks = [c for c in final_features if c in target_columns]
print("Label-derived features found:", label_leaks)

print("\n2. Baseline/product-derived feature check")
baseline_leaks = [
    c for c in final_features
    if any(word in c.lower() for word in baseline_keywords)
]
print("Baseline/product-derived features found:", baseline_leaks)

print("\n3. Obvious future/target-related feature-name check")
future_keywords = [
    "trend",
    "future",
    "next",
    "label",
    "outcome"
]

future_suspects = [
    c for c in final_features
    if any(word in c.lower() for word in future_keywords)
]

print("Future/target-related suspects:", future_suspects)

print("\n4. Target source")
print("is_declining_label = (trend_direction == 'down').astype(int)")

print("\nLeakage audit summary:")
print("Label-derived leakage:", "NONE" if not label_leaks else label_leaks)
print("Baseline/product leakage:", "NONE" if not baseline_leaks else baseline_leaks)
print("Feature-name future leakage:", "NONE" if not future_suspects else future_suspects)
print("Temporal window alignment: REQUIRES LIMITATION / MANUAL VERIFICATION")

Final model features:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'days_since_last_update', 'content_age_days', 'word_count']

1. Label-derived feature check
Label-derived features found: []

2. Baseline/product-derived feature check
Baseline/product-derived features found: []

3. Obvious future/target-related feature-name check
Future/target-related suspects: []

4. Target source
is_declining_label = (trend_direction == 'down').astype(int)

Leakage audit summary:
Label-derived leakage: NONE
Baseline/product leakage: NONE
Feature-name future leakage: NONE
Temporal window alignment: REQUIRES LIMITATION / MANUAL VERIFICATION


## 4. Claim rewrite

**Original claim:**

The Decision Tree can identify declining content using the available performance and content features.

**Safer claim:**

In this dataset, the Decision Tree measured an F1 score of 0.594 on the grouped-by-client test split. The model appeared to rely most heavily on impressions_90d and content_age_days, with additional use of avg_position and clicks_90d. These results provide directional, decision-support evidence for identifying patterns associated with the observed declining label. They do not establish that these features cause content to decline, and temporal alignment of the aggregate feature windows could not be fully verified from the starter dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.